In [3]:
import time

import bs4
import pandas as pd
import requests
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import gnews
import yfinance as yf
from gnews import GNews
from lxml.html.diff import href_token
import bs4 as bs

In [2]:
news=GNews(start_date=(2019,11,4), end_date=(2024,11,2),max_results=1000)
ns=news.get_news("Microsoft")

C:\Users\Asus\anaconda3\Lib\site-packages\gnews\gnews.py:218: UserWarning: Only searches using get_news support date ranges. Start and end dates will be ignored.
  return self._get_news_more_than_100(key)


In [1]:
news_df=pd.DataFrame(data=ns)
news_df.head()

NameError: name 'pd' is not defined

In [4]:
news_df.shape

(1000, 5)

In [40]:
temp=news_df
temp["full_text"] = (temp["title"].fillna("") + "" +temp["description"].fillna(""))

In [41]:
temp.head()

,title,description,published date,url,publisher,full_text
0,Microsoft Bans Term ‘Microslop’ From Official ...,Microsoft Bans Term ‘Microslop’ From Official ...,"Mon, 02 Mar 2026 19:40:56 GMT",https://news.google.com/rss/articles/CBMikwFBV...,"{'href': 'https://gizmodo.com', 'title': 'Gizm...",Microsoft Bans Term ‘Microslop’ From Official ...
1,"Microsoft says stop calling it Microslop, or y...","Microsoft says stop calling it Microslop, or y...","Mon, 02 Mar 2026 15:35:00 GMT",https://news.google.com/rss/articles/CBMiowFBV...,"{'href': 'https://www.pcworld.com', 'title': '...","Microsoft says stop calling it Microslop, or y..."
2,Why the Microsoft 365 Copilot bug matters for ...,Why the Microsoft 365 Copilot bug matters for ...,"Mon, 02 Mar 2026 15:37:32 GMT",https://news.google.com/rss/articles/CBMihwFBV...,"{'href': 'https://www.foxnews.com', 'title': '...",Why the Microsoft 365 Copilot bug matters for ...
3,Microsoft ends data center non-disclosure agre...,Microsoft ends data center non-disclosure agre...,"Tue, 03 Mar 2026 19:22:00 GMT",https://news.google.com/rss/articles/CBMi6wFBV...,"{'href': 'https://www.detroitnews.com', 'title...",Microsoft ends data center non-disclosure agre...
4,Microsoft’s big developer conference returns t...,Microsoft’s big developer conference returns t...,"Tue, 03 Mar 2026 17:00:00 GMT",https://news.google.com/rss/articles/CBMiggFBV...,"{'href': 'https://www.theverge.com', 'title': ...",Microsoft’s big developer conference returns t...


In [48]:
temp.iloc[2,1]

'Why the Microsoft 365 Copilot bug matters for data security  Fox News'

In [8]:
news_df.to_csv(r"C:\Users\Asus\Documents\Skills\Python\Sentiment Analysis\MSFT_news.csv")

In [4]:
subreddit=["https://www.reddit.com/r/wallstreetbets/","https://www.reddit.com/r/stocks/"]

In [12]:
def scrape_reddit(subreddit):
    all_data=[]
    for index,url in enumerate(subreddit):
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}
        sr_name=url.split("/")[-2]
        print(sr_name)
        try:
            response=requests.get(url,headers=headers,timeout=15)
            response.raise_for_status()

            soup=bs4.BeautifulSoup(response.content,"html.parser")

            subreddit_data={"name":sr_name,"url":url,"title":soup.title.string if soup.title.string else "No Title"}
            print(subreddit_data)

            topics=[]

            for heading in soup.find_all(['h1','h2','h3','h4']):
                text=heading.get_text(strip=True)

                if text and len(text)>3:
                    if any(keyword in text.lower() for keyword in ['rise','oil','bullish','stocks','bonds','tax','chips','luxury']):
                        topics.append({"title":text,"subreddit":sr_name,"type":"topic"})
            discussions=[]
            seen_url=set()

            for link in soup.find_all('a',href=True):
                text=link.get_text(strip=True)
                href=link['href']
                if text and len(text)>1 and ('/comments/' in href) and (href not in seen_url):
                    seen_url.add(href)
                    discussions.append({"title":text,"url":href,"type":"discussions"})

            subreddit_data["topic"]=topics
            subreddit_data["discussions"]=discussions
            print(discussions)

            all_data.append(subreddit_data)
            time.sleep(2)

        except Exception as e:
            print(e)

    return all_data



In [13]:
a=scrape_reddit(subreddit)

{'name': 'stocks', 'url': 'https://www.reddit.com/r/stocks/', 'title': 'Reddit - The heart of the internet'}
[{'title': 'Rate My Portfolio - r/Stocks Quarterly Thread March 2026votes •comment', 'url': '/r/stocks/comments/1rhtjtk/rate_my_portfolio_rstocks_quarterly_thread_march/', 'type': 'discussions'}, {'title': 'r/Stocks Daily Discussion Wednesday - Mar 04, 2026votes •comments', 'url': '/r/stocks/comments/1rkfnrv/rstocks_daily_discussion_wednesday_mar_04_2026/', 'type': 'discussions'}, {'title': 'Korean Stock Markets and Futures are crashing due to the Iran War', 'url': '/r/stocks/comments/1rka9et/korean_stock_markets_and_futures_are_crashing_due/', 'type': 'discussions'}, {'title': 'Trump’s Global Tariffs Struck Down by US Supreme Court', 'url': '/r/stocks/comments/1r9xpm4/trumps_global_tariffs_struck_down_by_us_supreme/', 'type': 'discussions'}, {'title': 'Trump Announces US will stop all trade with Spain. Spanish ADRs dip.', 'url': '/r/stocks/comments/1rjyvkm/trump_announces_us_wi

In [26]:
a[0]

{'name': 'wallstreetbets',
 'url': 'https://www.reddit.com/r/wallstreetbets/',
 'title': 'Reddit - The heart of the internet',
 'topic': [],
 'discussions': [{'title': 'What Are Your Moves Tomorrow, March 04, 2026votes •comments',
   'url': '/r/wallstreetbets/comments/1rk1aji/what_are_your_moves_tomorrow_march_04_2026/',
   'type': 'discussions'},
  {'title': 'Daily Discussion Thread for March 04, 2026',
   'url': '/r/wallstreetbets/comments/1rkjmj2/daily_discussion_thread_for_march_04_2026/',
   'type': 'discussions'},
  {'title': 'Dubai stock market crashes 4.6% at open.',
   'url': '/r/wallstreetbets/comments/1rke6hz/dubai_stock_market_crashes_46_at_open/',
   'type': 'discussions'},
  {'title': 'Supreme Court rules that Trump’s sweeping emergency tariffs are illegal | CNN Politics',
   'url': '/r/wallstreetbets/comments/1r9xu7b/supreme_court_rules_that_trumps_sweeping/',
   'type': 'discussions'},
  {'title': 'Hard Drive Bull Market',
   'url': '/r/wallstreetbets/comments/1rk77mo/h

ValueError: All arrays must be of the same length